# DenseNet — Densely Connected Convolutional Networks (2016)

_Papers · Computer Vision_

**Feed every layer the concatenation of ALL earlier feature maps, so features are reused and gradients reach every layer.**

---

This notebook is a practice scaffold for the **DenseNet — Densely Connected Convolutional Networks (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)

# --- 0. Sanity-check the worked example: channel growth k0 + k*(ell-1), k0=1, k=6, L=4. ---
k0, k, L = 1, 6, 4
ins = [k0 + k * (ell - 1) for ell in range(1, L + 1)]
print("layer input channels:", ins, " block output:", k0 + k * L)
# layer input channels: [1, 7, 13, 19]  block output: 25


# --- 1. The composite function H_ell: BN -> ReLU -> Conv(3x3), outputs exactly k channels. ---
def H(cin, kk):
    return nn.Sequential(nn.BatchNorm2d(cin), nn.ReLU(),
                         nn.Conv2d(cin, kk, 3, padding=1))


# --- 2. The dense block (built by hand). dense=True -> concat all; dense=False -> plain ablation. ---
class Block(nn.Module):
    def __init__(self, k0, k, L, dense=True):
        super().__init__()
        self.dense = dense
        self.layers = nn.ModuleList()
        cin = k0
        for _ in range(L):
            self.layers.append(H(cin, k))      # in-channels = k0 + k*(ell-1) when dense
            cin = (cin + k) if dense else k     # dense: pool grows by k; plain: next sees only k
        self.out_ch = cin

    def forward(self, x):
        if self.dense:
            feats = [x]
            for lyr in self.layers:
                feats.append(lyr(torch.cat(feats, dim=1)))   # Eqn. 2: H([x_0,...,x_{l-1}])
            return torch.cat(feats, dim=1)                   # output = all maps, k0 + k*L channels
        else:
            for lyr in self.layers:                          # ablation: each layer sees only the prev
                x = lyr(x)
            return x


# --- 3. A tiny classifier: stem -> block -> global average pool -> linear head. ---
class Net(nn.Module):
    def __init__(self, dense, k0=1, k=4, L=12, n_classes=3):
        super().__init__()
        self.stem  = nn.Conv2d(1, k0, 3, padding=1)
        self.block = Block(k0, k, L, dense)
        self.head  = nn.Linear(self.block.out_ch, n_classes)

    def forward(self, x):
        x = self.block(self.stem(x))
        x = x.mean(dim=(2, 3))                  # global average pool
        return self.head(x)


# --- 4. A small toy image dataset (swap in MNIST/CIFAR via torchvision in Colab; toy here for speed). ---
g = torch.Generator().manual_seed(1)
N, K = 600, 3
base = torch.randn(K, 1, 8, 8, generator=g)            # one 8x8 pattern per class
y = torch.randint(0, K, (N,), generator=g)
X = base[y] + 0.6 * torch.randn(N, 1, 8, 8, generator=g)


def train(dense, steps=120, lr=0.012):
    torch.manual_seed(0)
    net = Net(dense); net.train()
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lf  = nn.CrossEntropyLoss(); curve = []
    for t in range(steps):
        opt.zero_grad(); loss = lf(net(X), y); loss.backward(); opt.step()
        curve.append(loss.item())
    return curve, net.block.out_ch

dense_curve, dch = train(dense=True)
plain_curve, pch = train(dense=False)
print("dense block output channels:", dch, " | plain:", pch)
print("DENSE final loss:", round(dense_curve[-1], 4))
print("PLAIN final loss:", round(plain_curve[-1], 4), " (chance = ln 3 =", round(float(np.log(K)), 4), ")")
# With a deep, thin stack the dense block trains (loss falls) while the matched plain stack
# stalls near chance -- the vanishing-gradient effect, cured by dense connectivity.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_With a deep, thin stack, does dense connectivity let the net train while a matched plain stack stalls?_

In [ ]:
import torch, torch.nn as nn, numpy as np

# Two matched deep thin nets, identical except for dense connectivity (concat all earlier maps).
# No BatchNorm, so the deep plain stack suffers the vanishing gradient the paper describes.
torch.manual_seed(0)
g = torch.Generator().manual_seed(1)
N, K, k0, k, L = 600, 3, 1, 4, 12
base = torch.randn(K, 1, 8, 8, generator=g)
y = torch.randint(0, K, (N,), generator=g)
X = base[y] + 0.6 * torch.randn(N, 1, 8, 8, generator=g)

def H(cin, kk):
    return nn.Sequential(nn.ReLU(), nn.Conv2d(cin, kk, 3, padding=1))

class Block(nn.Module):
    def __init__(self, dense):
        super().__init__()
        self.dense = dense; self.layers = nn.ModuleList(); cin = k0
        for _ in range(L):
            self.layers.append(H(cin, k)); cin = (cin + k) if dense else k
        self.out_ch = cin
    def forward(self, x):
        if self.dense:
            feats = [x]
            for lyr in self.layers: feats.append(lyr(torch.cat(feats, 1)))
            return torch.cat(feats, 1)
        for lyr in self.layers: x = lyr(x)
        return x

class Net(nn.Module):
    def __init__(self, dense):
        super().__init__()
        self.stem = nn.Conv2d(1, k0, 3, padding=1)
        self.block = Block(dense); self.head = nn.Linear(self.block.out_ch, K)
    def forward(self, x):
        return self.head(self.block(self.stem(x)).mean(dim=(2, 3)))

def train(dense, steps=120, lr=0.012):
    torch.manual_seed(0)
    net = Net(dense); net.train()
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lf = nn.CrossEntropyLoss(); losses = []
    for t in range(steps):
        opt.zero_grad(); loss = lf(net(X), y); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

plain = train(dense=False)
dense = train(dense=True)
idx = np.linspace(0, 119, 30).astype(int)
print("Plain:", [[int(i), round(plain[i], 4)] for i in idx])
print("Dense:", [[int(i), round(dense[i], 4)] for i in idx])
# Plain -> stalls at ~1.098 (ln 3, chance). Dense -> falls to ~0.39. Only difference: concat all earlier maps.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation (dense vs plain). You have a working deep, thin dense block whose training loss
            falls. Turn it into a matched plain stack &mdash; same depth, same per-layer width &mdash; by
            feeding each layer only the previous layer's output instead of the concatenation. Retrain. What
            happens to the training curve, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one thing: each layer's input from torch.cat(feats, 1) to feats[-1] (and shrink the conv's in-channels from $k_0+k(\ell-1)$ to $k$). Keep depth, $k$, optimizer, data, and seed identical. — _An honest ablation changes only the dense connectivity, so any difference is attributable to it._
- Retrain the deep plain stack and watch its training loss: it stays flat near chance (for $C$ classes, $\ln C$), while the dense block's loss falls. — _Without direct short paths to early layers, the gradient vanishes through the deep plain stack, so it cannot learn; dense connectivity keeps the gradient flowing._
- Conclude that the dense connectivity &mdash; not extra width or depth &mdash; is what made the deep net trainable. — _Both stacks have the same depth and per-layer width; only the densely connected one optimizes, isolating connectivity as the cause._

**Answer:** The deep plain stack's training loss stays pinned near chance ($\ln C$ for $C$ classes),
                 while the dense block's loss falls steadily. Since the two differ only in whether each layer
                 sees the concatenation of all earlier maps, this isolates dense connectivity as the reason the
                 deep net trains: it is a gradient-flow / optimization fix (short paths to every layer),
                 not a width or parameter effect. The CODEVIZ panel shows exactly this contrast.

</details>

**Problem 2.** Channel bookkeeping. A dense block has input channels $k_0 = 3$ (an RGB map) and growth rate
            $k = 12$, with $L = 5$ layers. How many input channels does layer $4$ see, and how many channels
            does the whole block output?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply $k_0 + k(\ell-1)$ with $\ell = 4$: $3 + 12(4-1) = 3 + 36 = 39$. — _Layer $4$ sees the input plus the $k$ maps from each of the $3$ earlier layers: $3 + 12 + 12 + 12 = 39$._
- Block output = concatenation of the input and all $L$ layers: $k_0 + kL = 3 + 12\cdot 5 = 63$. — _Every layer added $k=12$ to the shared pool, and the input's $3$ channels are kept too._
- Note each conv still outputs only $12$ channels &mdash; thin layers, $63$-channel pooled output. — _That thinness with reuse is the source of DenseNet's parameter efficiency._

**Answer:** Layer $4$ sees $k_0 + k(\ell-1) = 3 + 12\cdot 3 = \mathbf{39}$ input channels. The block outputs
                 $k_0 + kL = 3 + 12\cdot 5 = \mathbf{63}$ channels, even though each convolution only ever produces
                 $12$. Thin layers, rich concatenated output &mdash; the reuse that lets a small $k$ go far.

</details>

**Problem 3.** Concatenation vs summation. ResNet writes $x_\ell = H_\ell(x_{\ell-1}) + x_{\ell-1}$ and
            DenseNet writes $x_\ell = H_\ell([x_0,\ldots,x_{\ell-1}])$. Suppose $x_{\ell-1}$ has $16$ channels and
            $H_\ell$ produces $16$ channels in ResNet's case, and $k=16$ in DenseNet's. How many channels come out
            of each, and what is the qualitative difference?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- ResNet: $H_\ell(x_{\ell-1})$ is $16$ channels and $x_{\ell-1}$ is $16$ channels; the element-wise sum keeps $16$ channels. — _Addition requires equal shapes and returns the same shape &mdash; the channel count does not grow._
- DenseNet: the output is the concatenation of all earlier maps; this layer alone adds its $k=16$ maps to the growing pool, and the block keeps growing. — _Concatenation stacks tensors, so channels accumulate rather than staying fixed._
- Note the information difference: addition can let features cancel; concatenation preserves every feature for later reuse. — _That preservation is exactly DenseNet's "feature reuse" claim._

**Answer:** ResNet's block outputs $16$ channels (the sum has the same shape as its inputs).
                 DenseNet's layer contributes $16$ new channels to a pool that keeps growing by $k$
                 each layer. Qualitatively: summation collapses two tensors into one fixed-size tensor (features can
                 cancel), while concatenation keeps every feature side by side (feature reuse, growing channels).
                 Same "shorter paths" cure, different operator.

</details>